# ACE Infrastructure — Production

Provisions all persistent Databricks resources required by ACE Chat.
Run this notebook **once** before any other notebook in this folder.

| Cell | Purpose |
|------|---------|
| 1 | Install `databricks-vectorsearch` |
| 2 | Central config — catalog, schema, table and index names |
| 3 | Store Azure SP credentials in Databricks Secrets (`ace-secrets`) |
| 4 | Create `policy_documents` Delta table |
| 5 | Create `inference_logs` Delta table |
| 5b | Migrate feedback columns — drop `text_assessments_array`, add `answer_feedback` + `citation_feedback` |
| 6 | Create `session_names` Delta table |
| 7 | Migration note — old log tables left intact (informational, nothing to run) |
| 8 | Create Vector Search endpoint |
| 9 | Create Vector Search delta-sync index |
| 10 | Grant Unity Catalog permissions to the service principal |

> **Prod note:** Verification cell removed. After Cell 10 completes, confirm resources are live in the Databricks UI before running `pdf_ingestion.ipynb`.


In [ ]:
# ================================================================
# CELL 1  -- Install Dependencies
# ================================================================
#
# PURPOSE:
#   Installs the `databricks-vectorsearch` Python package, which
#   provides the VectorSearchClient used in Cells 7, 8, and 10
#   to create and manage the Vector Search endpoint and index.
#
# WHY THIS PACKAGE:
#   Databricks does not include the Vector Search client in the
#   default cluster runtime. It must be installed manually before
#   any VS operations can be performed.
#
# KERNEL RESTART:
#   `dbutils.library.restartPython()` is called immediately after
#   install. This is required so that the newly installed package
#   is picked up by the Python interpreter. After the restart,
#   execution continues from Cell 2  -- do NOT re-run Cell 1.
#
# RUN ORDER:
#   Run this cell first, wait for the kernel to restart, then
#   proceed to Cell 2. Do not run Cell 1 again after restart.
# ================================================================

%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
# ================================================================
# CELL 2  -- Central Configuration
# ================================================================
#
# PURPOSE:
#   Defines all resource names and settings in one place.
#   Every subsequent cell references these variables  -- no resource
#   name is hardcoded anywhere else in this notebook.
#
# HOW TO USE:
#   If the catalog, schema, or any endpoint/index name needs to
#   change, update ONLY this cell. All downstream cells will
#   automatically pick up the new names.
#
# VARIABLE REFERENCE:
#
#   CATALOG           -- Unity Catalog top-level namespace.
#                      "ace_theorem" is the dedicated catalog for
#                      the ACE product. All ACE resources live here.
#
#   SCHEMA            -- Schema (database) within the catalog.
#                      "chat" groups all chat-related tables and
#                      models together.
#
#   TABLE_POLICY      -- Fully-qualified name of the Delta table that
#                      stores chunked PDF content (the RAG knowledge
#                      base). Written by 02_pdf_ingestion, read by
#                      the RAG agent via Vector Search.
#
#   TABLE_LOG         -- Fully-qualified name of the unified monitoring
#                      Delta table. Every conversation turn produces
#                      one row here. Written by the Supervisor agent
#                      at serving time via the SQL Statement API.
#
#   VS_ENDPOINT_NAME  -- Name of the Databricks Vector Search endpoint.
#                      This is the shared serverless compute that
#                      hosts the similarity search index. One endpoint
#                      can host multiple indexes.
#
#   VS_INDEX_NAME     -- Fully-qualified name of the Vector Search index
#                      built on top of policy_documents. The RAG agent
#                      calls similarity_search() against this index to
#                      retrieve the most relevant policy chunks for a
#                      given question.
#
#   EMBEDDING_MODEL   -- The Databricks Foundation Model endpoint used
#                      to generate embeddings when the VS index is
#                      synced. "databricks-gte-large-en" is a
#                      high-quality general-purpose English text
#                      embedding model available on all workspaces.
# ================================================================

CATALOG          = "ace_theorem"
SCHEMA           = "chat"

TABLE_POLICY     = f"{CATALOG}.{SCHEMA}.policy_documents"
TABLE_LOG           = f"{CATALOG}.{SCHEMA}.inference_logs"
TABLE_SESSION_NAMES = f"{CATALOG}.{SCHEMA}.session_names"  # Session names for history UX

VS_ENDPOINT_NAME = "ace-chat-vs-endpoint"
VS_INDEX_NAME    = f"{CATALOG}.{SCHEMA}.policy_documents_index"
EMBEDDING_MODEL  = "databricks-gte-large-en"
SQL_WAREHOUSE_ID = "b8583158cc1cf9e3"    # Used to grant the SP CAN_USE in Cell 9

print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")
print(f"Policy table : {TABLE_POLICY}")
print(f"Log table    : {TABLE_LOG}")
print(f"Session names: {TABLE_SESSION_NAMES}")
print(f"VS endpoint  : {VS_ENDPOINT_NAME}")
print(f"VS index     : {VS_INDEX_NAME}")

In [ ]:
# ================================================================
# CELL 3  -- Store Service Principal Credentials in Databricks Secrets
# ================================================================
#
# PURPOSE:
#   Stores the Azure AD service principal (SP) credentials that the
#   RAG agent needs to authenticate to Databricks Vector Search at
#   runtime inside the Model Serving container.
#
# WHY SECRETS ARE NEEDED:
#   When the RAG agent runs inside a Databricks Model Serving
#   container, it cannot use your personal notebook session token
#   to call Vector Search. Instead, it authenticates as a service
#   principal using MSAL client_credentials flow. The three values
#   it needs (client ID, secret, tenant ID) are stored in a
#   Databricks Secret scope so they can be injected as environment
#   variables when the serving endpoint is deployed (see Cell 5 of
#   07_serving_endpoint.ipynb).
#
# SECRET SCOPE: "ace-secrets"
#   A logical namespace for ACE-related secrets. Stored in
#   Databricks' encrypted secret backend  -- values are never
#   readable after being written (only usable via dbutils.secrets).
#
# KEYS STORED:
#   sp-client-id      -- The application (client) ID of the Azure SP.
#                      Already known: fb330a4b-c2bb-48b9-926c-b33a3650e1b7
#   sp-client-secret  -- The SP's client secret (password). Paste from
#                      Azure Portal *' App Registrations *' Certificates & Secrets.
#   sp-tenant-id      -- Your Azure AD tenant ID. Found in Azure Portal
#                      *' Azure Active Directory *' Overview.
#
# BEFORE RUNNING:
#   Replace "<paste the SP client secret here>" and
#   "<paste your Azure AD tenant ID here>" with real values.
#   The guard clause at the top will raise an error if you forget.
#
# IDEMPOTENT:
#   The Secrets API PUT operation overwrites existing values, so
#   this cell is safe to re-run if credentials rotate.
# ================================================================

import requests

# "" Authenticate using the current interactive notebook session token.
# This token has permission to manage secrets in this workspace.
# It is only used here during setup  -- NOT at serving time.
_nb_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_host     = spark.conf.get("spark.databricks.workspaceUrl")
_base_url = f"https://{_host}/api/2.0/secrets"
_headers  = {"Authorization": f"Bearer {_nb_token}"}

# FILL THESE IN before running
# SP_CLIENT_ID is already known and hardcoded below.
# You must paste in SP_CLIENT_SECRET and SP_TENANT_ID.
SP_CLIENT_ID     = "fb330a4b-c2bb-48b9-926c-b33a3650e1b7"  # ACE service principal app ID
SP_CLIENT_SECRET = "<paste the SP client secret here>"      # From Azure Portal
SP_TENANT_ID     = "<paste your Azure AD tenant ID here>"   # Azure AD tenant GUID
SECRET_SCOPE     = "ace-secrets"
#
# Guard clause  -- prevents accidentally running with placeholder values.
if "<paste" in SP_CLIENT_SECRET or "<paste" in SP_TENANT_ID:
    raise ValueError("Fill in SP_CLIENT_SECRET and SP_TENANT_ID before running this cell.")

# "" Step 1: Create the secret scope.
# initial_manage_principal="users" allows all workspace users to
# manage this scope. Adjust to a tighter group if needed.
# Returns RESOURCE_ALREADY_EXISTS if scope was previously created  -- that's fine.
resp = requests.post(f"{_base_url}/scopes/create",
    headers=_headers,
    json={"scope": SECRET_SCOPE, "initial_manage_principal": "users"}
)
if resp.status_code == 200:
    print(f"  [OK]   Secret scope '{SECRET_SCOPE}' created")
elif "RESOURCE_ALREADY_EXISTS" in resp.text:
    print(f"  [OK]   Secret scope '{SECRET_SCOPE}' already exists")
else:
    raise RuntimeError(f"Scope creation failed: {resp.status_code} {resp.text}")

# "" Step 2: Write each credential to the secret scope.
# The Secrets PUT API overwrites the value if the key already exists,
# making this step fully idempotent.
secrets = [
    ("sp-client-id",     SP_CLIENT_ID),
    ("sp-client-secret", SP_CLIENT_SECRET),
    ("sp-tenant-id",     SP_TENANT_ID),
]
for key, value in secrets:
    resp = requests.post(f"{_base_url}/put",
        headers=_headers,
        json={"scope": SECRET_SCOPE, "key": key, "string_value": value}
    )
    if resp.status_code == 200:
        print(f"  [OK]   {SECRET_SCOPE}/{key} stored")
    else:
        raise RuntimeError(f"Failed to store {key}: {resp.status_code} {resp.text}")

# "" Step 3: Verify that all three keys are present.
# The list endpoint only returns key names  -- values are always
# redacted by Databricks and can never be read back once stored.
resp = requests.get(f"{_base_url}/list",
    headers=_headers,
    params={"scope": SECRET_SCOPE}
)
stored_keys = [s["key"] for s in resp.json().get("secrets", [])]
for key in ["sp-client-id", "sp-client-secret", "sp-tenant-id"]:
    status = "[OK]" if key in stored_keys else "[MISSING]"
    print(f"  {status}  {SECRET_SCOPE}/{key}")

print("\nSecrets stored. Proceed to Cell 4.")

In [ ]:
# ================================================================
# CELL 4  -- Create the policy_documents Delta Table
# ================================================================
#
# PURPOSE:
#   Creates the RAG knowledge base table. This table stores lending
#   policy documents that have been chunked into smaller passages by
#   02_pdf_ingestion.ipynb. The RAG agent retrieves relevant chunks
#   from this table (via the Vector Search index) to ground its
#   answers in authoritative policy text.
#
# SCHEMA:
#   doc_id      -- Stable, unique identifier per chunk. Generated as
#                an MD5 hash of (doc_name + section + chunk_index)
#                by the ingestion notebook. Used as the primary key
#                for the Vector Search delta-sync index.
#
#   doc_name    -- Human-readable source document name (e.g.
#                "ACE_Lending_Policy_2024.pdf"). Included in RAG
#                citations returned to the loan officer.
#
#   section     -- The section heading under which this chunk appears
#                (e.g. "4.2.1 Loan-to-Value Requirements"). Included
#                in citations so officers can find the source quickly.
#
#   chunk_text  -- The actual text content of the chunk. This is the
#                column that gets embedded into vectors and searched
#                at query time.
#
# WHY CHANGE DATA FEED:
#   `delta.enableChangeDataFeed = true` is a hard requirement for
#   Databricks Vector Search delta-sync indexes. The VS engine reads
#   the CDC log to know which rows were added, updated, or deleted
#   since the last sync  -- this is what makes incremental re-embedding
#   possible. Without CDF enabled, the index creation in Cell 8
#   will fail with a configuration error.
#
# IDEMPOTENT:
#   CREATE TABLE IF NOT EXISTS  -- safe to re-run, no data loss.
# ================================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_POLICY} (
        doc_id     STRING NOT NULL,   -- MD5-based unique chunk identifier (VS primary key)
        doc_name   STRING NOT NULL,   -- Source PDF filename (used in citations)
        section    STRING NOT NULL,   -- Section heading the chunk belongs to (used in citations)
        chunk_text STRING NOT NULL    -- The actual text content that gets embedded and searched
    )
    USING DELTA
    -- Change Data Feed is REQUIRED by Databricks Vector Search for delta-sync indexes.
    -- It enables incremental re-embedding: only changed rows are re-processed on each sync.
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

print(f"Table '{TABLE_POLICY}' ready.")

In [ ]:
# ================================================================
# CELL 5  -- Create the inference_logs Delta Table
# ================================================================
#
# PURPOSE:
#   Creates the unified monitoring table for ACE Chat. Every time a
#   loan officer sends a message, the Supervisor agent writes one row
#   here (via a background daemon thread using the SQL Statement API).
#   This table is the source of truth for all analytics, health checks,
#   latency monitoring, error tracking, and audit trails.
#
# SCHEMA DESIGN NOTES:
#
#   Partitioning
#   PARTITIONED BY (request_date) enables efficient time-range queries.
#   All analytics queries filter by date so partition pruning dramatically
#   reduces the amount of data scanned.
#
#   Identity Column
#   id BIGINT GENERATED ALWAYS AS IDENTITY is an auto-incremented surrogate
#   key. Never set this manually -- Delta will reject any INSERT that tries
#   to supply a value.
#
#   Column Reference
#   request_date            -- DATE partition column (UTC date of request)
#   databricks_request_id   -- UUID per predict() call
#   client_request_id       -- session_id from Blazor (groups turns)
#   request_time            -- UTC start of predict()
#   status_code             -- HTTP-style: 200 = success, 500 = error
#   execution_duration_ms   -- Total predict() wall-clock time in ms
#   request                 -- Full request JSON (prompt, history, etc.)
#   response                -- Full response JSON (answer, intent, etc.)
#   type                    -- Intent: RAG | PAGE | BOTH | NONE
#   Question                -- Convenience copy of prompt
#   Answer                  -- Convenience copy of final answer text
#   requester               -- user_upn from user_identity
#   feedback                -- mirrors answer_feedback (primary overall signal)
#   answer_feedback         -- "up" / "down" on the answer quality
#   citation_feedback       -- "up" / "down" on citations; NULL for PAGE responses
#
# IDEMPOTENT:
#   CREATE TABLE IF NOT EXISTS  -- safe to re-run, no data loss.
# ================================================================

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE_LOG} (

        -- Partitioning & identity
        request_date           DATE        NOT NULL,
        id                     BIGINT      GENERATED ALWAYS AS IDENTITY,

        -- Request identification
        databricks_request_id  STRING      NOT NULL,
        client_request_id      STRING,
        request_id             STRING,

        -- Timing
        request_time           TIMESTAMP   NOT NULL,
        extracted_datetime_str STRING,

        -- HTTP-style status
        status_code            INT,
        status_response        STRING,
        logging_error_codes    STRING,

        -- Performance
        execution_duration_ms  BIGINT,
        sampling_fraction      DOUBLE,

        -- Full request / response blobs
        request                STRING,
        response               STRING,

        -- Routing metadata
        type                   STRING,
        served_entity_id       STRING,
        endpoint               STRING,
        filename               STRING,

        -- Convenience denormalized columns (fast queries, no JSON parse)
        Question               STRING,
        Answer                 STRING,
        requester              STRING,

        -- Feedback columns (NULL at write time; populated by the thumbs UI)
        feedback               STRING,   -- mirrors answer_feedback; primary overall signal
        answer_feedback        STRING,   -- "up" / "down" on the answer quality
        citation_feedback      STRING,   -- "up" / "down" on citations; NULL for PAGE responses
        timestamp_feedback     TIMESTAMP,
        feedback_clean         STRING
    )
    USING DELTA
    PARTITIONED BY (request_date)
''')

print(f"Table '{TABLE_LOG}' ready.")


In [ ]:
# ================================================================
# CELL 5b  -- Migrate inference_logs Feedback Columns (US4)
# ================================================================
#
# PURPOSE:
#   Drops text_assessments_array and adds answer_feedback +
#   citation_feedback to inference_logs.
#
#   Uses a check-first pattern: DESCRIBE TABLE first, then only
#   run ALTER TABLE for columns that actually need changing.
#   This avoids any errors from re-running on an already-migrated table.
#
# IDEMPOTENT: safe to re-run at any time.
# ================================================================

TABLE_LOG_FULL = f"{CATALOG}.{SCHEMA}.inference_logs"

# Read current schema once -- all decisions are based on this
current_cols = {r.col_name for r in spark.sql(f"DESCRIBE TABLE {TABLE_LOG_FULL}").collect()}

# Step 1: Drop text_assessments_array only if it still exists
if "text_assessments_array" in current_cols:
    spark.sql(f"ALTER TABLE {TABLE_LOG_FULL} DROP COLUMN text_assessments_array")
    print("  [OK]   Dropped text_assessments_array")
else:
    print("  [SKIP] text_assessments_array already absent")

# Step 2: Add answer_feedback only if missing
if "answer_feedback" not in current_cols:
    spark.sql(f"ALTER TABLE {TABLE_LOG_FULL} ADD COLUMNS (answer_feedback STRING)")
    print("  [OK]   Added answer_feedback")
else:
    print("  [SKIP] answer_feedback already present")

# Step 3: Add citation_feedback only if missing
if "citation_feedback" not in current_cols:
    spark.sql(f"ALTER TABLE {TABLE_LOG_FULL} ADD COLUMNS (citation_feedback STRING)")
    print("  [OK]   Added citation_feedback")
else:
    print("  [SKIP] citation_feedback already present")

# Verify final state
final_cols = {r.col_name for r in spark.sql(f"DESCRIBE TABLE {TABLE_LOG_FULL}").collect()}
print()
for col in ["feedback", "answer_feedback", "citation_feedback", "timestamp_feedback", "feedback_clean"]:
    print(f"  {'[OK]' if col in final_cols else '[MISSING]'}   {col}")
print(f"  {'[OK]' if 'text_assessments_array' not in final_cols else '[PRESENT - unexpected]'}   text_assessments_array absent")
print("\nFeedback schema migration complete.")


In [ ]:
# ================================================================
# CELL 6  -- Create the session_names Delta Table
# ================================================================
#
# PURPOSE:
#   Creates the session names lookup table for the ACE Chat sessions
#   history UX. When a user sends their first message of the day, the
#   Supervisor agent calls Haiku to generate a short 5-7 word title for
#   the session and writes it here via a background daemon thread.
#
# RELATIONSHIP TO inference_logs:
#   inference_logs  -- one row per turn (many rows per session)
#   session_names   -- one row per session (one per calendar day per user)
#
#   The sessions list query (GetSessionListAsync) LEFT JOINs these two
#   tables on client_request_id = session_id and uses:
#     COALESCE(session_name, FIRST(Question))
#   so sessions always have a display name even if Haiku failed.
#
# SCHEMA:
#   session_id    -- Blazor GUID == client_request_id in inference_logs.
#                  One per calendar day per user. Persisted in browser
#                  localStorage under key: ace-session-{oid}-{yyyy-MM-dd}.
#   requester     -- UPN of the user (user_upn from user_identity).
#   session_name  -- 5-7 word Haiku-generated title. Written once, never
#                  updated. COALESCE falls back to FIRST(Question).
#   created_at    -- UTC timestamp of the first turn (audit only).
#
# WRITE PATTERN:
#   Written by _save_session_name() in 05_supervisor_agent.ipynb.
#   Fire-and-forget daemon thread, wait_timeout="0s"  -- never blocks
#   the response stream. Only fires when is_first_turn is True.
#
# IDEMPOTENT: CREATE TABLE IF NOT EXISTS  -- safe to re-run.
# ================================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_SESSION_NAMES} (
        session_id   STRING    NOT NULL,  -- Blazor GUID == client_request_id in inference_logs
        requester    STRING    NOT NULL,  -- UPN of the user
        session_name STRING    NOT NULL,  -- Haiku-generated 5-7 word title
        created_at   TIMESTAMP NOT NULL   -- UTC timestamp of first turn
    )
    USING DELTA
    TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
""")

print(f"Table '{TABLE_SESSION_NAMES}' ready.")

## Cell 7  -- Migration Note: Old Log Tables

### Context
Before the current unified `inference_logs` table was introduced, the system used **two separate tables**:
- `ace_theorem.chat.inference_logs_input`  -- stored the inbound request fields (prompt, session ID, user identity, page context)
- `ace_theorem.chat.inference_logs_output`  -- stored the outbound response fields (answer, agent used, latency, token count)

These were joined at query time on `session_id` to reconstruct a full conversation turn, which was fragile and inconvenient.

### What Changed
The two tables were replaced with `inference_logs` (Cell 5), which captures the complete request + response in a single row, aligned with the Databricks native inference table schema.

### Why the Old Tables Were Not Dropped
**The old tables are intentionally left intact.** Dropping them would permanently destroy historical conversation data. They remain in place for:
- Historical reference and trend analysis
- Regulatory audit requirements
- Migration or backfill if needed in the future

### No Action Required
This cell is informational only  -- there is nothing to run. Skip to Cell 7.

### Querying Historical Data (if needed)
```sql
-- All historical input logs (pre-migration)
SELECT * FROM ace_theorem.chat.inference_logs_input ORDER BY timestamp DESC;

-- All historical output logs (pre-migration)
SELECT * FROM ace_theorem.chat.inference_logs_output ORDER BY timestamp DESC;

-- Reconstruct a full turn from old tables
SELECT i.prompt, i.user_upn, o.response, o.latency_ms
FROM ace_theorem.chat.inference_logs_input  i
JOIN ace_theorem.chat.inference_logs_output o ON i.session_id = o.session_id
ORDER BY i.timestamp DESC;
```

In [ ]:
# ================================================================
# CELL 8  -- Create the Vector Search Endpoint
# ================================================================
#
# PURPOSE:
#   Creates the Databricks Vector Search endpoint  -- the shared
#   serverless compute cluster that physically hosts the similarity
#   search index. The endpoint must exist before any index can be
#   created on it (Cell 8).
#
# WHAT A VS ENDPOINT IS:
#   A Vector Search endpoint is persistent, managed infrastructure
#   that stays alive between queries. It handles embedding lookups,
#   ANN (approximate nearest neighbour) searches, and serves
#   similarity_search() requests from the RAG agent at inference time.
#   One endpoint can host multiple indexes  -- if you add more document
#   collections in the future, they share this same endpoint.
#
# ENDPOINT TYPE: "STANDARD"
#   The standard type supports delta-sync indexes (which this project
#   uses). It provides automatic scaling and is suitable for
#   production workloads.
#
# TIMING:
#   A new VS endpoint takes approximately 5 minutes to provision.
#   This cell polls every 30 seconds and blocks until the state
#   transitions to "ONLINE" before returning.
#
# IDEMPOTENT:
#   If the endpoint already exists, the VectorSearchClient raises
#   an "already exists" exception which is caught and logged  -- the
#   cell continues normally without re-creating the endpoint.
#
# STATES WHILE PROVISIONING:
#   PROVISIONING / PENDING *' the endpoint is being created
#   ONLINE                 *' ready to host indexes
#   Any other state        *' unexpected  -- raises RuntimeError
# ================================================================

import time
from databricks.vector_search.client import VectorSearchClient

# Initialise the VS client using the current session's credentials.
vsc = VectorSearchClient()

# "" Create the endpoint (idempotent).
# endpoint_type="STANDARD" is required for delta-sync indexes.
try:
    vsc.create_endpoint(name=VS_ENDPOINT_NAME, endpoint_type="STANDARD")
    print(f"Endpoint '{VS_ENDPOINT_NAME}' creation started...")
except Exception as e:
    if "already exists" in str(e).lower():
        # Endpoint was created in a previous run  -- nothing to do.
        print(f"Endpoint '{VS_ENDPOINT_NAME}' already exists  -- skipping.")
    else:
        raise  # Unexpected error  -- surface it immediately

# "" Poll until the endpoint is ONLINE.
# Provisioning typically takes ~5 minutes. We check every 30 seconds.
# PROVISIONING and PENDING are expected transient states.
print("Waiting for endpoint to come ONLINE (this takes ~5 minutes)...")
while True:
    status = vsc.get_endpoint(VS_ENDPOINT_NAME)["endpoint_status"]["state"]
    print(f"  Status: {status}")
    if status == "ONLINE":
        print("Endpoint is ONLINE.")
        break
    elif status in ("PROVISIONING", "PENDING"):
        time.sleep(30)
    else:
        raise RuntimeError(f"Endpoint entered unexpected state: {status}")

In [ ]:
# ================================================================
# CELL 9  -- Create the Vector Search Delta-Sync Index
# ================================================================
#
# PURPOSE:
#   Creates the Vector Search index over the `policy_documents` table.
#   This index is what the RAG agent searches at inference time  --
#   it converts a loan officer's question into an embedding vector
#   and returns the most semantically similar policy chunks.
#
# WHAT A DELTA-SYNC INDEX IS:
#   A delta-sync index is tightly coupled to a Delta table (source).
#   When you trigger a sync, Databricks reads new/changed rows from
#   the table's Change Data Feed, generates embeddings for the
#   specified column (`chunk_text`), and updates the ANN index.
#   This means the index stays in sync with the policy documents
#   without requiring a full re-index every time.
#
# KEY PARAMETERS:
#   endpoint_name                 -- The VS endpoint created in Cell 7.
#   index_name                    -- Fully-qualified Unity Catalog name.
#   source_table_name             -- policy_documents table (must have
#                                  CDF enabled  -- done in Cell 4).
#   pipeline_type = "TRIGGERED"   -- Sync only runs when explicitly
#                                  triggered (e.g. after new PDFs are
#                                  ingested in 02_pdf_ingestion).
#                                  The alternative "CONTINUOUS" would
#                                  sync automatically but incurs
#                                  higher compute cost for a document
#                                  set that changes infrequently.
#   primary_key = "doc_id"        -- The column that uniquely identifies
#                                  each row. Used to apply CDC deltas
#                                  (inserts, updates, deletes).
#   embedding_source_column       -- The text column to embed.
#                                  `chunk_text` is the policy passage.
#   embedding_model_endpoint_name -- The Foundation Model endpoint used
#                                  to generate embeddings. Must match
#                                  the model used at query time so
#                                  distances are meaningful.
#
# TIMING:
#   First-time index creation takes 8c, not "15 minutes because it must
#   embed every existing row in policy_documents from scratch.
#   Subsequent syncs (after adding new PDFs) are incremental and
#   typically complete in under 2 minutes.
#
# IDEMPOTENT:
#   If the index already exists, VectorSearchClient raises an
#   "already exists" exception which is caught and ignored.
# ================================================================

# "" Create the delta-sync index (idempotent).
try:
    vsc.create_delta_sync_index(
        endpoint_name                 = VS_ENDPOINT_NAME,
        index_name                    = VS_INDEX_NAME,
        source_table_name             = TABLE_POLICY,
        pipeline_type                 = "TRIGGERED",    # Manual sync  -- matches infrequent PDF updates
        primary_key                   = "doc_id",       # MD5-based unique chunk ID (set in 02_pdf_ingestion)
        embedding_source_column       = "chunk_text",   # The text passage column to embed
        embedding_model_endpoint_name = EMBEDDING_MODEL,  # databricks-gte-large-en
    )
    print(f"Index '{VS_INDEX_NAME}' creation started...")
except Exception as e:
    if "already exists" in str(e).lower():
        # Index was created in a previous run  -- nothing to do.
        print(f"Index '{VS_INDEX_NAME}' already exists  -- skipping.")
    else:
        raise  # Unexpected error  -- surface it immediately

# "" Poll until the index is ONLINE and ready.
# First run takes 8c, not "15 minutes (embeds all existing rows).
# We check `status.ready` rather than a string state because the
# detailed_state values vary across Databricks versions.
print("Waiting for index to come ONLINE (this takes 8-15 minutes on first run)...")
while True:
    idx        = vsc.get_index(VS_ENDPOINT_NAME, VS_INDEX_NAME)
    idx_status = idx.describe().get("status", {})
    detailed   = idx_status.get("detailed_state", "")  # e.g. "ONLINE_NO_PENDING_UPDATE"
    ready      = idx_status.get("ready", False)
    print(f"  {detailed}  |  ready={ready}")
    if ready:
        print("Index is ONLINE and ready.")
        break
    time.sleep(30)

In [ ]:
# ================================================================
# CELL 10  -- Grant Service Principal Permissions
# ================================================================
#
# PURPOSE:
#   Grants the ACE service principal (SP) the minimum Unity Catalog
#   permissions it needs to operate inside the Model Serving container
#   at runtime. Without these grants the serving container will receive
#   PERMISSION_DENIED errors when the RAG agent tries to query Vector
#   Search or when the Supervisor tries to write inference logs.
#
# WHY A SERVICE PRINCIPAL:
#   Databricks Model Serving containers do not run as your personal
#   user. The RAG agent authenticates as the ACE service principal
#   (using the SP credentials stored in Cell 3) to call Vector Search.
#   The SP needs explicit Unity Catalog grants for every resource it
#   accesses  -- it does not inherit any user permissions.
#
# UNITY CATALOG GRANT HIERARCHY:
#   Unity Catalog enforces grants top-down. You cannot grant access
#   to a table without also granting USE on the parent catalog and
#   schema. The order of grants here is:
#     1. USE CATALOG   -- allows the SP to see the catalog exists
#     2. USE SCHEMA    -- allows the SP to see the schema exists
#     3. SELECT / MODIFY on specific tables and indexes
#
# GRANTS EXPLAINED:
#   USE CATALOG ace_theorem            -- SP can navigate the catalog
#   USE SCHEMA ace_theorem.chat        -- SP can navigate the schema
#   SELECT ON policy_documents         -- RAG agent reads policy chunks
#                                       (via VS similarity_search, which
#                                       reads the source table on sync)
#   SELECT ON inference_logs           -- Required for query validation
#                                       before INSERT at serving time
#   MODIFY ON inference_logs           -- Supervisor can INSERT log rows
#                                       via the SQL Statement API
#   SELECT ON policy_documents_index   -- RAG agent calls
#                                       similarity_search() on this index
#
# MINIMUM PRIVILEGE:
#   No WRITE or CREATE grants are given on policy_documents  --
#   the SP cannot modify the knowledge base at serving time.
#   No admin or metastore-level grants are given.
#
# IDEMPOTENT:
#   "already granted" errors are caught and logged as [OK]  --
#   safe to re-run without duplicating grants.
# ================================================================

# Read the SP's application ID from secrets.
# This ensures the grant target is always consistent with Cell 3.
SP_APP_ID = dbutils.secrets.get(scope="ace-secrets", key="sp-client-id")

# Ordered list of grant statements.
# Must be top-down: catalog *' schema *' tables/indexes.
grants = [
    # Unity Catalog hierarchy (required before any table/index grants)
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{SP_APP_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{SP_APP_ID}`",

    # policy_documents  -- RAG agent reads chunks via VS at inference time
    f"GRANT SELECT ON TABLE {CATALOG}.{SCHEMA}.policy_documents TO `{SP_APP_ID}`",

    # inference_logs  -- Supervisor writes one row per turn at serving time
    f"GRANT SELECT ON TABLE {CATALOG}.{SCHEMA}.inference_logs TO `{SP_APP_ID}`",  # For validation
    f"GRANT MODIFY ON TABLE {CATALOG}.{SCHEMA}.inference_logs TO `{SP_APP_ID}`",  # For INSERT

    # session_names  -- Supervisor writes one row per session (first turn only)
    f"GRANT SELECT ON TABLE {CATALOG}.{SCHEMA}.session_names TO `{SP_APP_ID}`",   # For validation
    f"GRANT MODIFY ON TABLE {CATALOG}.{SCHEMA}.session_names TO `{SP_APP_ID}`",   # For INSERT

    # VS index  -- RAG agent calls similarity_search() at inference time
    f"GRANT SELECT ON TABLE {CATALOG}.{SCHEMA}.policy_documents_index TO `{SP_APP_ID}`",

]

# SQL warehouse CAN_USE  -- must be set via REST API, not spark.sql()
# Warehouse permissions are not Unity Catalog objects; spark.sql() cannot grant
# them. Use the Databricks Permissions API instead.
import requests as _req
_host  = spark.conf.get("spark.databricks.workspaceUrl", "").rstrip("/")
_host  = f"https://{_host}" if not _host.startswith("http") else _host
_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_r = _req.patch(
    f"{_host}/api/2.0/permissions/warehouses/{SQL_WAREHOUSE_ID}",
    headers={"Authorization": f"Bearer {_token}", "Content-Type": "application/json"},
    json={"access_control_list": [{"service_principal_name": SP_APP_ID, "permission_level": "CAN_USE"}]}
)
if _r.ok:
    print(f"  [OK]   CAN_USE ON SQL WAREHOUSE {SQL_WAREHOUSE_ID}")
else:
    print(f"  [FAIL] CAN_USE ON SQL WAREHOUSE {SQL_WAREHOUSE_ID}: {_r.text}")

print("=" * 55)
print("ACE Infrastructure  -- SP Permission Grants")
print("=" * 55)
print(f"  SP app ID : {SP_APP_ID}")
print()

for stmt in grants:
    try:
        spark.sql(stmt)
        action = stmt.split("GRANT ")[1].split(" TO")[0]
        print(f"  [OK]   {action}")
    except Exception as e:
        if "already exists" in str(e).lower() or "already granted" in str(e).lower():
            # Grant already applied in a previous run  -- idempotent, no action needed.
            action = stmt.split("GRANT ")[1].split(" TO")[0]
            print(f"  [OK]   {action} (already granted)")
        else:
            # Unexpected error  -- log it but continue so all grants are attempted.
            print(f"  [FAIL] {stmt[:80]}")
            print(f"         {e}")

print()
print("Permissions granted. Proceed to Cell 10 (Verification).")